# Lista de Exercícios — Volatilidade Implícita

**Entrega individual (notebook).**

Este notebook resolve 10 exercícios sobre **volatilidade implícita** usando o modelo de **Black–Scholes** e dois métodos numéricos de inversão: **Newton–Raphson** e **Bisseção**.

## Sumário
1. Volatilidade implícita de uma call (Newton-Raphson vs Bisseção)
2. Volatilidade implícita de uma put
3. Dados reais (Yahoo Finance) + preço sintético + recuperação por Bisseção
4. Cinco strikes e construção do *volatility smile*
5. Comparação Newton-Raphson vs Bisseção
6. Sensibilidade do Newton-Raphson ao chute inicial
7. Limites de arbitragem e inexistência de vol. implícita
8. Opção muito fora do dinheiro, Vega e dificuldades do Newton-Raphson
9. PETR4, VALE3 e AAPL — histórica vs implícita
10. Análise de mercado — opção cara ou barata?

## 0. Importações e funções base

Definimos aqui o modelo de Black–Scholes, o Vega e os dois algoritmos de inversão que serão reutilizados em todos os exercícios.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.set_printoptions(precision=6, suppress=True)
plt.rcParams['figure.figsize'] = (9, 5)

# ----------------------------------------------------------------------
# Modelo de Black-Scholes
# ----------------------------------------------------------------------
def d1_d2(S, K, r, T, sigma, q=0.0):
    """Calcula d1 e d2 do modelo de Black-Scholes (q = dividend yield)."""
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

def bs_price(S, K, r, T, sigma, option='call', q=0.0):
    """Preco de uma opcao europeia (call ou put) por Black-Scholes."""
    d1, d2 = d1_d2(S, K, r, T, sigma, q)
    if option == 'call':
        return S * np.exp(-q*T) * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)
    elif option == 'put':
        return K * np.exp(-r*T) * norm.cdf(-d2) - S * np.exp(-q*T) * norm.cdf(-d1)
    else:
        raise ValueError("option deve ser 'call' ou 'put'")

def bs_vega(S, K, r, T, sigma, q=0.0):
    """Vega: derivada do preco em relacao a sigma (igual para call e put)."""
    d1, _ = d1_d2(S, K, r, T, sigma, q)
    return S * np.exp(-q*T) * norm.pdf(d1) * np.sqrt(T)

print('Funcoes de Black-Scholes definidas.')

In [ ]:
# ----------------------------------------------------------------------
# Newton-Raphson para volatilidade implicita
# ----------------------------------------------------------------------
def iv_newton(price_market, S, K, r, T, option='call', q=0.0,
              sigma0=0.2, tol=1e-8, max_iter=100):
    """Volatilidade implicita por Newton-Raphson.
    Retorna (sigma, n_iteracoes, historico_sigma).
    Retorna sigma=NaN se nao convergir.
    """
    sigma = sigma0
    hist = [sigma]
    for i in range(1, max_iter + 1):
        preco = bs_price(S, K, r, T, sigma, option, q)
        vega = bs_vega(S, K, r, T, sigma, q)
        diff = preco - price_market
        if abs(diff) < tol:
            return sigma, i, hist
        if vega < 1e-12:           # vega quase nulo -> passo instavel
            return np.nan, i, hist
        sigma = sigma - diff / vega
        if sigma <= 0:             # sigma negativo nao tem sentido
            sigma = 1e-4
        hist.append(sigma)
    return np.nan, max_iter, hist   # nao convergiu

# ----------------------------------------------------------------------
# Bissecao para volatilidade implicita
# ----------------------------------------------------------------------
def iv_bisection(price_market, S, K, r, T, option='call', q=0.0,
                 low=1e-4, high=5.0, tol=1e-8, max_iter=200):
    """Volatilidade implicita por Bissecao.
    Retorna (sigma, n_iteracoes). Retorna NaN se o preco estiver fora
    do intervalo de precos atingivel [low, high].
    """
    p_low = bs_price(S, K, r, T, low, option, q) - price_market
    p_high = bs_price(S, K, r, T, high, option, q) - price_market
    if p_low * p_high > 0:           # nao ha mudanca de sinal -> sem raiz
        return np.nan, 0
    for i in range(1, max_iter + 1):
        mid = 0.5 * (low + high)
        p_mid = bs_price(S, K, r, T, mid, option, q) - price_market
        if abs(p_mid) < tol or (high - low) / 2 < tol:
            return mid, i
        if p_low * p_mid < 0:
            high = mid
            p_high = p_mid
        else:
            low = mid
            p_low = p_mid
    return 0.5 * (low + high), max_iter

print('Metodos de inversao (Newton-Raphson e Bissecao) definidos.')

---
## Exercício 1 — Call europeia: Newton-Raphson vs Bisseção

Dados: `S0 = 100`, `K = 100`, `r = 10% a.a.`, `T = 0.5 ano`, preço de mercado = `12`.

In [ ]:
S0, K, r, T, preco_mkt = 100, 100, 0.10, 0.5, 12.0

iv_nr, it_nr, hist_nr = iv_newton(preco_mkt, S0, K, r, T, 'call', sigma0=0.20)
iv_bs, it_bs = iv_bisection(preco_mkt, S0, K, r, T, 'call')

print(f"Newton-Raphson: sigma = {iv_nr:.6f}  ({iv_nr*100:.4f}%) em {it_nr} iteracoes")
print(f"Bissecao      : sigma = {iv_bs:.6f}  ({iv_bs*100:.4f}%) em {it_bs} iteracoes")
print(f"Diferenca absoluta entre os metodos: {abs(iv_nr - iv_bs):.2e}")

# Verificacao: reprecificar com a vol encontrada deve devolver ~12
print(f"\nPreco reconstruido (NR): {bs_price(S0, K, r, T, iv_nr, 'call'):.6f}")
print(f"Preco reconstruido (Bi): {bs_price(S0, K, r, T, iv_bs, 'call'):.6f}")

**Comentário.** Os dois métodos convergem para essencialmente a **mesma** volatilidade implícita (diferença da ordem de $10^{-8}$, controlada apenas pela tolerância). O Newton-Raphson precisa de pouquíssimas iterações (convergência quadrática), enquanto a Bisseção precisa de mais passos (convergência linear), mas é robusta porque sempre mantém a raiz dentro de um intervalo com mudança de sinal.

---
## Exercício 2 — Put europeia (preço de mercado = 8)

Mesmos parâmetros, agora uma **put** com preço de mercado = 8.

In [ ]:
preco_put = 8.0

iv_nr_p, it_nr_p, _ = iv_newton(preco_put, S0, K, r, T, 'put', sigma0=0.20)
iv_bs_p, it_bs_p = iv_bisection(preco_put, S0, K, r, T, 'put')

print(f"Newton-Raphson: sigma = {iv_nr_p:.6f}  ({iv_nr_p*100:.4f}%) em {it_nr_p} iteracoes")
print(f"Bissecao      : sigma = {iv_bs_p:.6f}  ({iv_bs_p*100:.4f}%) em {it_bs_p} iteracoes")

# Limites de arbitragem da put europeia:
#   max(K e^{-rT} - S, 0)  <=  put  <=  K e^{-rT}
lim_inf = max(K*np.exp(-r*T) - S0, 0)
lim_sup = K*np.exp(-r*T)
print(f"\nLimites de arbitragem da put: [{lim_inf:.4f}, {lim_sup:.4f}]")
print(f"Preco de mercado = {preco_put} esta dentro dos limites? "
      f"{lim_inf <= preco_put <= lim_sup}")

**A volatilidade implícita encontrada faz sentido?**

Sim. O preço de mercado (8) está dentro dos limites de não-arbitragem da put (acima do valor intrínseco descontado e abaixo de $Ke^{-rT}$), portanto **existe** uma volatilidade positiva única que reproduz esse preço. O valor obtido é positivo e de magnitude econômica plausível (faixa típica de mercado). Note ainda que, pela **paridade put-call**, uma put a 8 corresponde a uma call sintética cujo preço é $P + S_0 - Ke^{-rT}$, e por isso a vol. implícita da put e da call de mesmo strike devem coincidir.

---
## Exercício 3 — Dados reais + preço sintético + recuperação por Bisseção

Escolhemos uma ação brasileira, estimamos `S0` e a **volatilidade histórica** (anualizada a partir dos log-retornos diários). Criamos um preço sintético de opção usando uma volatilidade **20% maior** que a histórica e recuperamos essa vol por Bisseção.

> O bloco abaixo tenta baixar dados via `yfinance`. Se não houver internet no ambiente, ele usa um **fallback** com valores plausíveis para que o notebook rode de ponta a ponta.

In [ ]:
def baixar_dados(ticker, period='1y'):
    """Retorna (S0, vol_historica_anualizada, fonte). Usa fallback se offline."""
    try:
        import yfinance as yf
        df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
        if df is None or df.empty:
            raise RuntimeError('sem dados')
        precos = df['Close'].dropna()
        # yfinance pode retornar coluna como DataFrame; garante serie 1D
        precos = np.asarray(precos).ravel().astype(float)
        log_ret = np.diff(np.log(precos))
        vol_hist = np.std(log_ret, ddof=1) * np.sqrt(252)
        S0 = float(precos[-1])
        return S0, float(vol_hist), 'yfinance'
    except Exception as e:
        # Fallback offline (valores ilustrativos)
        fallback = {
            'PETR4.SA': (38.5, 0.42),
            'VALE3.SA': (61.0, 0.38),
            'AAPL':     (195.0, 0.25),
        }
        S0, vol = fallback.get(ticker, (50.0, 0.30))
        return S0, vol, f'fallback ({type(e).__name__})'

ticker = 'PETR4.SA'
S0_real, vol_hist, fonte = baixar_dados(ticker)
print(f"Ticker: {ticker}  |  fonte: {fonte}")
print(f"S0 estimado        = {S0_real:.4f}")
print(f"Vol. historica     = {vol_hist:.4f}  ({vol_hist*100:.2f}% a.a.)")

# Preco sintetico com vol 20% maior
vol_sintetica = vol_hist * 1.20
K3, r3, T3 = round(S0_real), 0.10, 0.5   # ATM aproximado
preco_sint = bs_price(S0_real, K3, r3, T3, vol_sintetica, 'call')
print(f"\nVol. sintetica (+20%) = {vol_sintetica:.4f}  ({vol_sintetica*100:.2f}%)")
print(f"Strike K           = {K3}")
print(f"Preco sintetico    = {preco_sint:.4f}")

# Recuperacao por Bissecao
vol_rec, it_rec = iv_bisection(preco_sint, S0_real, K3, r3, T3, 'call')
print(f"\nVol. recuperada (Bissecao) = {vol_rec:.6f}  em {it_rec} iteracoes")
print(f"Erro vs vol sintetica      = {abs(vol_rec - vol_sintetica):.2e}")

**Comentário.** A Bisseção recupera com precisão a volatilidade que usamos para gerar o preço sintético — o erro fica limitado pela tolerância. Isso confirma que o mapeamento *preço → volatilidade* é bem definido e invertível na região de não-arbitragem (o preço da call é estritamente crescente em $\sigma$).

---
## Exercício 4 — Cinco strikes e o *volatility smile*

Strikes em 80%, 90%, 100%, 110% e 120% de `S0`. Atribuímos uma volatilidade **diferente** a cada strike (formato de sorriso: mais alta nas asas, mais baixa no centro), geramos preços sintéticos e **recuperamos** a vol implícita, reconstruindo o smile.

In [ ]:
S0_4, r4, T4 = 100.0, 0.10, 0.5
moneyness = np.array([0.80, 0.90, 1.00, 1.10, 1.20])
strikes = moneyness * S0_4

# Volatilidades 'verdadeiras' em formato de smile
vols_true = np.array([0.32, 0.26, 0.22, 0.25, 0.30])

precos_sint = np.array([bs_price(S0_4, K, r4, T4, sig, 'call')
                        for K, sig in zip(strikes, vols_true)])

# Recupera vol implicita por Newton-Raphson (com fallback p/ bissecao)
vols_impl = []
for K, p in zip(strikes, precos_sint):
    v, _, _ = iv_newton(p, S0_4, K, r4, T4, 'call', sigma0=0.20)
    if np.isnan(v):
        v, _ = iv_bisection(p, S0_4, K, r4, T4, 'call')
    vols_impl.append(v)
vols_impl = np.array(vols_impl)

print('Strike   Moneyness  Preco     Vol_true  Vol_impl')
for K, m, p, vt, vi in zip(strikes, moneyness, precos_sint, vols_true, vols_impl):
    print(f'{K:6.1f}   {m:6.2f}    {p:7.4f}   {vt:6.4f}   {vi:6.4f}')

plt.plot(strikes, vols_impl*100, 'o-', color='tab:blue', label='Vol. implicita recuperada')
plt.plot(strikes, vols_true*100, 'x--', color='tab:red', alpha=0.6, label='Vol. verdadeira')
plt.axvline(S0_4, color='gray', ls=':', label='ATM (S0)')
plt.xlabel('Strike (K)'); plt.ylabel('Volatilidade implicita (%)')
plt.title('Volatility Smile')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Comentário.** A vol implícita recuperada coincide com a vol verdadeira em cada strike, e o gráfico mostra o formato de **sorriso**: volatilidades mais altas para opções fora do dinheiro (asas) e mais baixas perto do *at-the-money*. Esse padrão é observado no mercado real e contradiz a hipótese de volatilidade constante do Black-Scholes.

---
## Exercício 5 — Comparação Newton-Raphson vs Bisseção

Comparamos os dois métodos em **número de iterações, estabilidade, velocidade e facilidade de implementação**, usando o caso do Exercício 1 e cronometrando.

In [ ]:
import timeit

args = (12.0, 100, 100, 0.10, 0.5, 'call')

_, it_nr, _ = iv_newton(*args, sigma0=0.20)
_, it_bs = iv_bisection(*args)

t_nr = timeit.timeit(lambda: iv_newton(*args, sigma0=0.20), number=2000) / 2000
t_bs = timeit.timeit(lambda: iv_bisection(*args), number=2000) / 2000

print(f"{'Criterio':<22}{'Newton-Raphson':<18}{'Bissecao'}")
print('-'*58)
print(f"{'Iteracoes':<22}{it_nr:<18}{it_bs}")
print(f"{'Tempo medio (us)':<22}{t_nr*1e6:<18.2f}{t_bs*1e6:.2f}")
print(f"{'Convergencia':<22}{'quadratica':<18}{'linear'}")
print(f"{'Precisa de Vega?':<22}{'sim':<18}{'nao'}")
print(f"{'Garante convergir?':<22}{'nao':<18}{'sim (se ha sinal)'}")

| Critério | Newton-Raphson | Bisseção |
|---|---|---|
| **Nº de iterações** | Poucas (convergência **quadrática**) | Muitas (convergência **linear**, ~$\log_2$ da precisão) |
| **Estabilidade** | Pode **divergir** se o Vega for pequeno (opções muito ITM/OTM) ou com chute ruim | **Sempre converge** quando há mudança de sinal no intervalo |
| **Velocidade** | Muito rápido **por iteração efetiva**, mas cada passo exige preço + Vega | Cada passo é barato (só o preço), porém são necessários mais passos |
| **Facilidade de implementação** | Exige a derivada (Vega) e tratamento de casos degenerados | Mais simples — só precisa do sinal da função |

**Conclusão.** Newton-Raphson é o método de escolha quando se tem um bom chute inicial e o Vega não é desprezível (caso típico ATM). A Bisseção é preferível quando se precisa de **robustez garantida** (opções extremas, dados ruidosos). Uma prática comum é o **híbrido**: tentar Newton-Raphson e cair para Bisseção quando ele falha — exatamente o que fizemos no Exercício 4.

---
## Exercício 6 — Sensibilidade do Newton-Raphson ao chute inicial

Testamos chutes iniciais de **10%, 30%, 80% e 150%** e verificamos se convergem para a mesma vol implícita (caso do Exercício 1).

In [ ]:
chutes = [0.10, 0.30, 0.80, 1.50]
print(f"{'Chute inicial':<15}{'Vol implicita':<18}{'Iteracoes':<12}{'Convergiu?'}")
print('-'*55)
historicos = {}
for c in chutes:
    v, it, hist = iv_newton(12.0, 100, 100, 0.10, 0.5, 'call', sigma0=c)
    historicos[c] = hist
    ok = 'sim' if not np.isnan(v) else 'NAO'
    vtxt = f'{v:.6f}' if not np.isnan(v) else 'NaN'
    print(f"{c*100:>6.0f}%{'':<8}{vtxt:<18}{it:<12}{ok}")

# Trajetorias de convergencia
for c, hist in historicos.items():
    plt.plot(range(len(hist)), np.array(hist)*100, 'o-', label=f'chute {c*100:.0f}%')
plt.xlabel('Iteracao'); plt.ylabel('sigma (%)')
plt.title('Convergencia do Newton-Raphson para diferentes chutes iniciais')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Comentário.** Para uma opção ATM, o preço é uma função **monótona e bem comportada** de $\sigma$, então o Newton-Raphson converge para a **mesma** volatilidade implícita a partir de todos os chutes — apenas o **número de iterações** muda (chutes muito distantes, como 150%, levam alguns passos a mais). Em opções extremas, porém, chutes ruins podem produzir passos grandes e até $\sigma$ negativo, daí a importância das salvaguardas implementadas (`sigma <= 0` e Vega mínimo).

---
## Exercício 7 — Limites de arbitragem e inexistência de vol. implícita

Escolhemos um preço de mercado **muito baixo** para uma call e verificamos os limites de não-arbitragem.

Para uma call europeia:
$$\max(S_0 - Ke^{-rT},\,0) \;\le\; C \;\le\; S_0.$$
O limite inferior corresponde a $\sigma \to 0$ e o superior a $\sigma \to \infty$.

In [ ]:
S7, K7, r7, T7 = 100, 100, 0.10, 0.5
lim_inf = max(S7 - K7*np.exp(-r7*T7), 0)   # preco com sigma -> 0
lim_sup = S7                                # preco com sigma -> infinito
print(f"Limites de nao-arbitragem da call: [{lim_inf:.4f}, {lim_sup:.4f}]")

preco_baixo = 2.0   # abaixo do limite inferior (~4.88)
print(f"\nPreco de mercado escolhido = {preco_baixo} (abaixo do limite inferior!)")

v_nr, it_nr7, _ = iv_newton(preco_baixo, S7, K7, r7, T7, 'call', sigma0=0.20)
v_bs, it_bs7 = iv_bisection(preco_baixo, S7, K7, r7, T7, 'call')
print(f"Newton-Raphson -> {v_nr}")
print(f"Bissecao       -> {v_bs}  (NaN = sem mudanca de sinal no intervalo)")

# Mostra que nenhum sigma produz preco tao baixo
sig_grid = np.linspace(1e-4, 3.0, 300)
precos = [bs_price(S7, K7, r7, T7, s, 'call') for s in sig_grid]
plt.plot(sig_grid*100, precos, label='Preco BS da call')
plt.axhline(preco_baixo, color='red', ls='--', label=f'Preco de mercado = {preco_baixo}')
plt.axhline(lim_inf, color='green', ls=':', label=f'Limite inferior = {lim_inf:.2f}')
plt.xlabel('sigma (%)'); plt.ylabel('Preco da call')
plt.title('Preco da call vs sigma — o preco de mercado e inatingivel')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Por que pode não existir volatilidade implícita válida?**

O preço de Black-Scholes de uma call é uma função **crescente e contínua** de $\sigma$, variando entre o valor intrínseco descontado ($\sigma \to 0$) e $S_0$ ($\sigma \to \infty$). Se o preço de mercado estiver **abaixo do limite inferior** (ou acima de $S_0$), **nenhum** valor de $\sigma \ge 0$ reproduz esse preço — o preço viola não-arbitragem. Nesse caso:
- a **Bisseção** retorna `NaN`, pois não há mudança de sinal no intervalo (comportamento correto e seguro);
- o **Newton-Raphson** tende a divergir ou empurrar $\sigma$ para zero/negativo, sem solução válida.

A inexistência de vol. implícita é, portanto, um **sinal de erro nos dados** (preço estale, cotação ruim) ou de oportunidade de arbitragem.

---
## Exercício 8 — Opção muito fora do dinheiro: Vega e dificuldade do Newton-Raphson

Escolhemos uma call **muito fora do dinheiro** (`K` bem acima de `S0`) e analisamos o Vega.

In [ ]:
S8, r8, T8 = 100, 0.10, 0.5
vol_real = 0.20

print(f"{'K':>6}{'Moneyness':>12}{'Preco call':>14}{'Vega':>12}")
print('-'*46)
for K8 in [100, 130, 160, 200]:
    p = bs_price(S8, K8, r8, T8, vol_real, 'call')
    vg = bs_vega(S8, K8, r8, T8, vol_real)
    print(f"{K8:>6}{K8/S8:>12.2f}{p:>14.6f}{vg:>12.6f}")

# Caso extremo: K = 200 (deep OTM)
K_otm = 200
preco_otm = bs_price(S8, K_otm, r8, T8, vol_real, 'call')
vega_otm = bs_vega(S8, K_otm, r8, T8, vol_real)
print(f"\nDeep OTM (K={K_otm}): preco = {preco_otm:.6e}, Vega = {vega_otm:.6e}")

v_nr, it_nr8, hist8 = iv_newton(preco_otm, S8, K_otm, r8, T8, 'call', sigma0=0.50)
v_bs, it_bs8 = iv_bisection(preco_otm, S8, K_otm, r8, T8, 'call')
print(f"Newton-Raphson (chute 50%): sigma = {v_nr}  em {it_nr8} it.")
print(f"Bissecao                  : sigma = {v_bs:.6f} em {it_bs8} it.")

**Por que o Newton-Raphson tem dificuldade?**

O passo do Newton-Raphson é $\sigma_{n+1} = \sigma_n - \dfrac{C(\sigma_n) - C_{mkt}}{\text{Vega}(\sigma_n)}$. Em opções **muito fora do dinheiro** o **Vega é minúsculo** (o preço quase não responde a $\sigma$). Dividir o resíduo por um Vega próximo de zero produz **passos enormes**, que podem:
- jogar $\sigma$ para valores negativos ou absurdamente altos;
- causar **oscilação ou divergência**.

Além disso, o preço é tão pequeno que erros de arredondamento ($10^{-6}$ ou menos) dominam, tornando a vol implícita mal-condicionada. A **Bisseção** é muito mais confiável nesse regime, pois não usa a derivada — por isso costuma ser preferida nas asas do smile.

---
## Exercício 9 — PETR4, VALE3 e AAPL: histórica vs implícita

Estimamos a volatilidade histórica das três ações, assumimos preços sintéticos de mercado (aqui com vol implícita = histórica × fator) e calculamos a vol implícita, comparando os resultados.

In [ ]:
tickers = ['PETR4.SA', 'VALE3.SA', 'AAPL']
# Fatores que simulam um 'premio de volatilidade' embutido no preco de mercado
fatores_mkt = {'PETR4.SA': 1.15, 'VALE3.SA': 0.95, 'AAPL': 1.25}
r9, T9 = 0.10, 0.5

print(f"{'Ticker':<11}{'S0':>9}{'Vol_hist':>10}{'Preco_mkt':>11}{'Vol_impl':>10}{'Dif%':>8}")
print('-'*60)
resultados = []
for tk in tickers:
    S0_t, vol_h, fonte = baixar_dados(tk)
    K_t = round(S0_t)                      # ATM
    vol_mkt_assumida = vol_h * fatores_mkt[tk]
    preco_mkt = bs_price(S0_t, K_t, r9, T9, vol_mkt_assumida, 'call')
    vol_i, _, _ = iv_newton(preco_mkt, S0_t, K_t, r9, T9, 'call', sigma0=0.30)
    if np.isnan(vol_i):
        vol_i, _ = iv_bisection(preco_mkt, S0_t, K_t, r9, T9, 'call')
    dif = (vol_i - vol_h) / vol_h * 100
    resultados.append((tk, S0_t, vol_h, vol_i, dif))
    print(f"{tk:<11}{S0_t:>9.2f}{vol_h:>10.4f}{preco_mkt:>11.4f}{vol_i:>10.4f}{dif:>7.1f}%")

# Grafico comparativo
nomes = [r[0] for r in resultados]
vh = [r[2]*100 for r in resultados]
vi = [r[3]*100 for r in resultados]
x = np.arange(len(nomes)); w = 0.35
plt.bar(x - w/2, vh, w, label='Vol. historica', color='tab:gray')
plt.bar(x + w/2, vi, w, label='Vol. implicita', color='tab:orange')
plt.xticks(x, nomes); plt.ylabel('Volatilidade (%)')
plt.title('Volatilidade historica vs implicita')
plt.legend(); plt.grid(alpha=0.3, axis='y'); plt.show()

**Comentário.** A vol implícita recuperada bate com a vol de mercado que assumimos (validação numérica). Na comparação:
- **PETR4** e **AAPL** têm vol implícita **acima** da histórica → o mercado precifica um *prêmio de risco/volatilidade* (expectativa de turbulência futura, demanda por proteção).
- **VALE3** aparece com vol implícita **abaixo** da histórica → o mercado espera uma volatilidade futura menor do que a observada no passado.

Diferenças entre histórica (olha para o **passado**) e implícita (olha para a **expectativa futura** precificada nas opções) são normais e contêm informação de mercado.

---
## Exercício 10 — Análise de mercado: opção cara ou barata?

Se a volatilidade implícita está muito **acima** da histórica, a opção parece cara ou barata? Discussão das possíveis razões.

In [ ]:
# Demonstracao quantitativa: mesmo strike, vol implicita > historica => premio maior
S10, K10, r10, T10 = 100, 100, 0.10, 0.5
vol_hist10 = 0.20
vol_impl10 = 0.35   # implicita bem acima da historica

preco_justo = bs_price(S10, K10, r10, T10, vol_hist10, 'call')  # 'valor justo' p/ vol historica
preco_mercado = bs_price(S10, K10, r10, T10, vol_impl10, 'call')
print(f"Preco com vol HISTORICA  ({vol_hist10*100:.0f}%): {preco_justo:.4f}")
print(f"Preco com vol IMPLICITA  ({vol_impl10*100:.0f}%): {preco_mercado:.4f}")
print(f"Sobrepreco (premio de volatilidade): {preco_mercado - preco_justo:.4f} "
      f"(+{(preco_mercado/preco_justo - 1)*100:.1f}%)")

### Discussão

**A opção parece CARA.** O preço de uma opção é estritamente crescente na volatilidade; logo, se a vol implícita (que o mercado está cobrando) é muito maior que a vol histórica (estimativa do risco efetivo do ativo), o prêmio pago está **acima** do que a dinâmica passada justificaria. Em termos de estratégia, isso favoreceria **vender** volatilidade (vender a opção) — desde que a vol histórica seja, de fato, um bom previsor da realizada.

**Possíveis razões para a vol implícita estar acima da histórica:**
1. **Expectativa de eventos futuros** — resultados trimestrais, eleições, decisões de juros, anúncios regulatórios. A histórica não "vê" o futuro; a implícita, sim.
2. **Prêmio de risco de volatilidade** — investidores pagam a mais por proteção (puts), elevando estruturalmente a implícita acima da realizada.
3. **Aversão a risco / demanda por hedge** — em momentos de incerteza, a procura por opções empurra os preços (e a implícita) para cima.
4. **Liquidez e oferta/demanda** — opções pouco líquidas ou desequilíbrio de fluxo distorcem preços.
5. **Assimetria (skew)** — puts OTM costumam ter implícita elevada por proteção contra quedas.

**Ressalvas importantes:** "cara" não significa lucro garantido ao vender. A vol implícita pode estar **correta** ao antecipar um aumento real de volatilidade; vender vol tem risco de cauda (perdas grandes se o evento se materializar). A comparação histórica × implícita é um **indicador**, não uma regra automática de negociação.

---
## Conclusão geral

- A **volatilidade implícita** é a vol que, inserida no Black-Scholes, reproduz o preço de mercado da opção. Como não há fórmula fechada, recorremos a métodos numéricos.
- **Newton-Raphson**: rápido (convergência quadrática), mas exige o Vega e pode falhar quando o Vega é pequeno (opções extremas) ou com chutes ruins.
- **Bisseção**: mais lenta, porém **robusta** — sempre converge quando há mudança de sinal no intervalo. Ideal nas asas do smile e com dados ruidosos.
- Os **limites de arbitragem** delimitam quando a vol implícita existe; fora deles, o preço é inconsistente.
- O **volatility smile** e as diferenças **histórica × implícita** mostram que o mercado real desvia das hipóteses do Black-Scholes e carrega informação sobre expectativas e prêmios de risco.